In [19]:
import numpy as np
import pandas as pd
from pathlib import Path

In [20]:
RESULTS_PATH = Path("../data/raw/IF_1872_2026/results.csv")

results = pd.read_csv(RESULTS_PATH)

results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [21]:
print(results.info())
print("---" * 30)
print(results.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 49477 entries, 0 to 49476
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49477 non-null  str    
 1   home_team   49477 non-null  str    
 2   away_team   49477 non-null  str    
 3   home_score  49433 non-null  float64
 4   away_score  49433 non-null  float64
 5   tournament  49477 non-null  str    
 6   city        49477 non-null  str    
 7   country     49477 non-null  str    
 8   neutral     49477 non-null  bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 3.1 MB
None
------------------------------------------------------------------------------------------
date           0
home_team      0
away_team      0
home_score    44
away_score    44
tournament     0
city           0
country        0
neutral        0
dtype: int64


In [22]:
results["date"] = pd.to_datetime(results["date"], format="%Y-%m-%d", errors="raise")

results.dtypes

date          datetime64[us]
home_team                str
away_team                str
home_score           float64
away_score           float64
tournament               str
city                     str
country                  str
neutral                 bool
dtype: object

In [23]:
TEAM_NAME_MAPPING = {
    "Turkey": "Türkiye",
    "United States": "USA",
    "Czech Republic": "Czechia",
}

results[["home_team", "away_team"]] = results[["home_team", "away_team"]].replace(TEAM_NAME_MAPPING)

In [24]:
teams_after_cleaning = set(results["home_team"]).union(results["away_team"])

[name for name in ["Turkey", "United States", "Czech Republic"] if name in teams_after_cleaning]

[]

In [25]:
[name for name in ["Türkiye", "USA", "Czechia"] if name in teams_after_cleaning]

['Türkiye', 'USA', 'Czechia']

In [26]:
score_columns = ["home_score", "away_score"]

complete_scores = results[score_columns].notna().all(axis=1)
missing_scores = results[score_columns].isna().all(axis=1)
partial_scores = results[score_columns].isna().any(axis=1) & ~missing_scores

assert not partial_scores.any(), "Rows with exactly one missing score found"

historical_matches = results.loc[complete_scores].copy()
future_matches = results.loc[missing_scores].copy()

print("historical_matches:", len(historical_matches))
print("future_matches:", len(future_matches))

historical_matches: 49433
future_matches: 44


In [27]:
historical_matches[score_columns] = historical_matches[score_columns].astype(int)

historical_matches[score_columns].dtypes

home_score    int64
away_score    int64
dtype: object

In [28]:
historical_matches["outcome"] = np.select(
    [
        historical_matches["home_score"] > historical_matches["away_score"],
        historical_matches["home_score"] < historical_matches["away_score"],
    ],
    ["home_win", "away_win"],
    default="draw",
)

historical_matches[["home_team", "away_team", "home_score", "away_score", "outcome"]].head()

,home_team,away_team,home_score,away_score,outcome
0,Scotland,England,0,0,draw
1,England,Scotland,4,2,home_win
2,Scotland,England,2,1,home_win
3,England,Scotland,2,2,draw
4,Scotland,England,3,0,home_win


In [29]:
historical_matches["outcome"].value_counts()

outcome
home_win    24227
away_win    13963
draw        11243
Name: count, dtype: int64

In [30]:
all_output_teams = (
    set(historical_matches["home_team"])
    | set(historical_matches["away_team"])
    | set(future_matches["home_team"])
    | set(future_matches["away_team"])
)

old_names = {"Turkey", "United States", "Czech Republic"}
canonical_names = {"Türkiye", "USA", "Czechia"}
valid_outcomes = {"home_win", "away_win", "draw"}

assert len(historical_matches) == 49_433
assert len(future_matches) == 44
assert len(historical_matches) + len(future_matches) == len(results)

assert historical_matches[score_columns].notna().all().all()
assert future_matches[score_columns].isna().all().all()
assert historical_matches[score_columns].dtypes.eq("int64").all()

assert historical_matches["outcome"].isin(valid_outcomes).all()
assert old_names.isdisjoint(all_output_teams), f"Old names still present: {old_names & all_output_teams}"
assert canonical_names.issubset(all_output_teams), f"Missing canonical names: {canonical_names - all_output_teams}"

print("All checks passed.")

All checks passed.
